# Python Testing Script
Use this template to create a Python testing script that imports your CRUD Python module to call and test the create and read instances of CRUD functionality. 
### 1. Import your CRUD Python module
### 2. Instantiate an instance of your class
### 3. Use your create function create a new record in the aac database
### 4. Use your read funtion to return records from the aac database 
### 5. Use your update function to update an existing record in the aac database
### 6. Use your delete function to delete an existing record in the aac database

<div class="alert alert-block alert-success"><span style="color:black;"><b>Tip: </b>You should have created test functionality for items 1 thru 4 during Module Four Milestone, copy over your existing test script code and expand it here to call and test the Update and Delete functionality.</span></div>

In [2]:
from pymongo import MongoClient 
from bson.objectid import ObjectId 
from urllib.parse import quote_plus

class AnimalShelter(object): 
    """ CRUD operations for Animal collection in MongoDB """ 

    def __init__(self, username, password): 
        # Connection Variables 
        print(f"Connecting with user: {username} and pass: {password}")
        USER = quote_plus(username) 
        PASS = quote_plus(password) 
        HOST = 'localhost' 
        PORT = 27017 
        DB = 'aac' 
        COL = 'animals' 
        # 
        # Initialize Connection 
        # 
        self.client = MongoClient(f'mongodb://{USER}:{PASS}@{HOST}:{PORT}/?authSource=admin') 
        self.database = self.client[DB] 
        self.collection = self.database[COL] 

    # Create a method to return the next available record number for use in the create method
            
    # Complete this create method to implement the C in CRUD. 
    def create(self, data):
        """
        Inserts a document into the specified MongoDB database and collection.
        """
        if data is not None and isinstance(data, dict):
            try:
                insert_result = self.collection.insert_one(data)
                #Check if the document was inserted successfully
                if insert_result.acknowledged:
                    return True
                else:
                    return False
            except Exception as e:
                print(f"An error occured during insert: {e}")
                return False
        else:
            raise Exception("Nothing to save, due to data parameters being empty or not a dictionary")
    def read(self, query=None):
        if query is not None and isinstance(query, dict):
            try:
                #Must use find(0 rather than find_one()
                cursor = self.collection.find(query)
                #Convert the cursor to a list and return
                return list(cursor)
            except Exception as e:
                print(f"An error occured during query: {e}")
                return []
        else:
            return []
    def update (self, query, new_data):
        if query is not None and isinstance(query, dict) and new_data is not None and isinstance(new_data, dict):
            try:
                #Use update_many to modify all matching documents
                result = self.collection.update_many(query, {"$set": new_data})
                return result.modified_count
            except Exception as e:
                print(f"An error occured during update: {e}")
                return 0
        else:
            raise Exception("Update failed: Search query and new data must be non empty dictionaries")
    def delete(self, query):
        if query is not None and isinstance(query, dict):
            try:
                result = self.collection.delete_many(query)
                return result.deleted_count
            except Exception as e:
                print(f"An error occured during delete: {e}")
                return 0
        else:
            raise Exception("Delete failed: Search query must be a non empty dictionary")

In [1]:
import importlib
import CRUD_Python_Module
importlib.reload(CRUD_Python_Module)

from CRUD_Python_Module import AnimalShelter

#Instantiate class with credentials set up
shelter = AnimalShelter('aacuser', 'password123')

#Test Create Functionality
sample_animal = {
    "animal_id": "A123456",
    "name": "TestDog",
    "animal_type": "Dog",
    "breed": "Labrador Retriever Mix",
    "color": "Black",
    "outcome_type": "Adoption"
}

is_created = shelter.create(sample_animal)
print(f"Create status: {is_created}")
#---------------------------------------------------
#Test Read
#---------------------------------------------------
results = shelter.read({"animal_id": "A123456"})
print(f"Read results count: {len(results)}")

#Print retrieved document to confirm data
for doc in results:
    print(doc)
#----------------------------------------------------
#Test UPDATE
#----------------------------------------------------
query = {"animal_id": "A123456"}
update_data = {"name": "UpdatedTestDog", "outcome_type": "Returned to Owner"}

update_count = shelter.update(query, update_data)
print(f"Update results count: {update_count}")

#Verify if update works
updated_results = shelter.read({"animal_id": "A123456"})
for doc in updated_results:
    print(f"Updated Name: {doc.get('name')}")
    
#------------------------------------------------------
#Test DELETEabs
#------------------------------------------------------
delete_count = shelter.delete({"animal_id": "A123456"})
print(f"Delete results count: {delete_count}")

#Verify item(s) was deleted
final_check = shelter.read({"animal_id": "A123456"})
print(f"After delete count results: {len(final_check)}")



Connecting with user: aacuser and pass: password123
Create status: True
Read results count: 1
{'_id': ObjectId('6a6ddef14b5ee299202b5981'), 'animal_id': 'A123456', 'name': 'TestDog', 'animal_type': 'Dog', 'breed': 'Labrador Retriever Mix', 'color': 'Black', 'outcome_type': 'Adoption'}
Update results count: 1
Updated Name: UpdatedTestDog
Delete results count: 1
After delete count results: 0
